[Homework 2]('https://colab.research.google.com/github/Jaguar838/llm-zoomcamp/blob/main/HW/2026/02-vector-search/hw-02.ipynb')

In [ ]:
# Install necessary Python packages for data processing, search, LLM interaction, environment variable management, and Git data reading.
%pip install requests minsearch groq gitsource sqlitesearch huggingface-hub tqdm onnxruntime tokenizers

## Homework: Vector Search

In this homework, we put what we learned in Module 2 into practice.

We'll first turn text into vectors, then search by similarity.
We'll also learn something new and see how to combine vector search with keyword search. We'll skip the RAG part and focus solely on search.

Like in homework 1, our knowledge base is the course lessons themselves.
Each module has a `lessons/` folder of numbered markdown
pages, and we pull them from GitHub. We use the same commit, `8c1834d`,
so everyone works with the exact same 72 pages.

> It's possible your answers won't match exactly. If so, select the closest one.

## Setup

In this homework we won't use the same approach for embedding as in the
module. That is, we won't use the sentence-transformers library. Instead,
we'll use the lightweight embedding approach with the ONNX `Embedder`.

Both approaches produce identical vectors, but the ONNX runtime is far
lighter. It needs no PyTorch and no CUDA, which makes the install about
30x smaller and lets it run anywhere, including a basic Codespace. We
skimmed through it in the lesson and said we'd cover it in the homework -
so here we are.

We prepare the environment the same way as in the module's
[ONNX Runtime](../../../02-vector-search/lessons/09-onnx-embedder.md)
lesson.

Create a fresh project and install the dependencies:

```bash
mkdir llm-zoomcamp-hw2 && cd llm-zoomcamp-hw2
uv init --no-workspace
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter
```

We also need two helper scripts from the `embed/` directory of the course
repo:

- [`download.py`](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/embed/download.py)
(fetches an ONNX model from HuggingFace) and
- [`embedder.py`](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/embed/embedder.py) (the `Embedder` class with an `encode` interface)

Let's download them:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
wget $PREFIX/download.py
wget $PREFIX/embedder.py
```

By default `download.py` fetches `Xenova/all-MiniLM-L6-v2`, the ONNX
version of the `all-MiniLM-L6-v2` model from the lessons:

```bash
uv run python download.py
```

Now we're ready to do the homework.

In [ ]:
%%bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
wget $PREFIX/download.py
wget $PREFIX/embedder.py

## Q1. Embedding a query

Embed the following query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value
(`v[0]`)?

* -0.31
* -0.02
* 0.12
* 0.44

## Loading the data

We pull the lesson pages from the course repository, the same way as in
homework 1. We pin to commit `8c1834d` so everyone works with the same
data.

```python
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
```

Each document is a dictionary with a `filename` and `content`, and there
are 72 pages.

In [ ]:
from gitsource import GithubRepositoryDataReader

# Initialize GithubRepositoryDataReader to read markdown files from a specific GitHub repository commit.
# It filters for files in the 'lessons/' path.
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

# Read the files matching the criteria.
files = reader.read()

In [ ]:
documents = []

# Iterate through the raw file data, parse each file into a document object,
# and store them in the 'documents' list.
for file in files:
    doc = file.parse()
    documents.append(doc)

# Display the first document to inspect its structure and content.
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [4]:
!python download.py

tokenizer.json: 100% 712k/712k [00:00<00:00, 126MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json
onnx/model.onnx: 100% 90.4M/90.4M [00:00<00:00, 98.8MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


In [7]:
from embedder import Embedder

model_path = 'models/Xenova/all-MiniLM-L6-v2'
embedder = Embedder(model_path)

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(f"Vector length: {len(v)}")
print(f"Q1: v[0] = {v[0]:.2f}")

Vector length: 384
Q1: v[0] = -0.02


## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two
of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed
its `content`, and compute the cosine similarity with the query vector
from Q1. What do you get?

* 0.07
* 0.37
* 0.68
* 0.92


In [9]:
from gitsource import GithubRepositoryDataReader
# 2. Get Query Vector (v) from Q1
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

# 3. Fetch the specific document content
# Note: We use the same commit and reader settings as in the notebook
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
    )

target_filename = '02-vector-search/lessons/07-sqlitesearch-vector.md'
documents = [file.parse() for file in reader.read()]

doc = next((d for d in documents if d['filename'] == target_filename), None)

if doc is None:
    print(f"Document {target_filename} not found!")

# 4. Embed the document content
v_doc = embedder.encode(doc['content'])

# 5. Compute cosine similarity (dot product since vectors are normalized)
similarity = v.dot(v_doc)

print(f"Target file: {target_filename}")
print(f"Q2: similarity: {similarity:.2f}")

Target file: 02-vector-search/lessons/07-sqlitesearch-vector.md
Q2: similarity: 0.36


In [ ]:
import time
from sqlitesearch import TextSearchIndex

index_sql = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="lesson_llm_course.db"
)

for doc in documents:
    index_sql.add(doc)
    print(f"""Added: {doc["filename"]}...""")
    time.sleep(0.5)

index_sql.close()
print("Done. Index saved to lesson_llm_course.db")

In [ ]:
index_sql.count()

72

In [ ]:
import os

file_path = 'lesson_llm_course.db'
file_size_bytes = os.path.getsize(file_path)
file_size_kb = file_size_bytes / 1024
file_size_mb = file_size_kb / 1024

print(f"File size of {file_path}: {file_size_mb:.2f} MB)")

File size of lesson_llm_course.db: 1.02 MB)


In [ ]:
results = index_sql.search(query, num_results=72)
sql_result_search = [doc["filename"] for doc in results]

In [ ]:
print(f"Count files: {len(sql_result_search)}")
print(f"Q2: {sql_result_search[0]}")

Count files: 59
Q2: 01-agentic-rag/lessons/14-agentic-loop.md


## Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

```python
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
```

We embed every chunk's `content` with `encode_batch`, stack the vectors
into a matrix `X`, and score the Q1 query against all chunks:

```python
scores = X.dot(v)
```

Which file does the highest-scoring chunk belong to (its `filename`)?

* `02-vector-search/lessons/03-embeddings-dataset.md`
* `02-vector-search/lessons/06-rag-vector.md`
* `02-vector-search/lessons/07-sqlitesearch-vector.md`
* `02-vector-search/lessons/09-onnx-embedder.md`

In [10]:
from gitsource import chunk_documents

# Chunk the documents into smaller pieces with a specified size and step for overlapping.
chunks = chunk_documents(documents, size=2000, step=1000)

In [11]:
# Print the total number of chunks created.
print(f"Q4: {len(chunks)}")

Q4: 295


In [13]:
import numpy as np

# Embed all chunks
embeddings = []
for chunk in chunks:
    embedding = embedder.encode(chunk['content'])
    embeddings.append(embedding)

# Stack embeddings into a NumPy array X
X = np.array(embeddings)

scores = X.dot(v)

In [17]:
scores

array([ 3.15187611e-01,  2.01479664e-01,  5.90559037e-02, -7.67661288e-02,
        1.18452466e-01, -1.41697012e-01, -2.81406495e-02, -4.65669117e-02,
       -2.06994543e-02, -6.07744147e-02,  2.13273769e-01,  8.87600958e-02,
       -1.97269268e-02,  3.11629985e-01,  5.51034635e-01,  2.04008152e-01,
        2.12515802e-01,  1.93649107e-01,  2.51961267e-01,  1.31078610e-01,
        2.59120607e-01,  7.63816369e-02,  9.59193203e-02,  9.81471228e-03,
       -3.59107168e-02,  1.01211406e-02,  1.10786940e-01, -9.90259915e-02,
       -3.71170645e-02,  7.59057333e-02, -3.35340234e-02,  8.86841484e-03,
        1.02636448e-01,  6.89615413e-02,  1.29408854e-01,  2.57709121e-01,
        3.23680576e-01,  1.06350076e-01,  5.61891208e-02,  2.34017441e-01,
        1.97954358e-01,  9.64296342e-02,  1.93709934e-01,  2.16719269e-01,
        3.48340450e-01,  5.10906541e-02,  2.05212833e-01,  1.05416182e-01,
       -3.25432660e-02,  4.94665347e-02,  2.38574873e-01, -3.44207304e-02,
        1.82165430e-01,  

In [14]:
max_score_index = np.argmax(scores)
highest_scoring_chunk = chunks[max_score_index]

print(f"Q3: {highest_scoring_chunk['filename']}")

Q3: 02-vector-search/lessons/07-sqlitesearch-vector.md


In [ ]:
import os
from groq import Groq
from google.colab import userdata

api_key_groq = userdata.get('GROQ_API_KEY')

# Initialize the Groq client with the retrieved API key.
client_groq = Groq(api_key=api_key_groq)

In [ ]:
# Send a chat completion request to the Groq API using the 'llama-3.1-8b-instant' model.
chat_completion = client_groq.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.1-8b-instant",
)

# Print the content of the assistant's response.
print(chat_completion.choices[0].message.content)

Fast language models have gained significant attention in the field of natural language processing (NLP) in recent years due to their ability to efficiently process and generate large volumes of text data. The importance of fast language models can be seen in several areas:

1. **Efficient Inference**: Fast language models can perform inference tasks much faster than traditional language models, which is crucial when dealing with large-scale applications such as chatbots, voice assistants, and text summarization systems. This efficiency enables faster response times, improved user experience, and increased scalability.
2. **Real-Time Processing**: Fast language models can process text data in real-time, allowing for applications such as live chat, sentiment analysis, and text classification to operate seamlessly. This is particularly important for applications that require timely responses to user input.
3. **Large-Scale Deployments**: Fast language models can be deployed at scale, ena

In [ ]:
# Define the system instructions for the RAG assistant.
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

# Define the prompt template that structures the question and context for the LLM.
PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-21 05:56:18--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-21 05:56:18 (27.9 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [ ]:
from dataclasses import dataclass
from rag_helper import RAGBase as OriginalRAGBase

# Define a dataclass to store the result of a RAG query, including the answer and token counts.
@dataclass
class RAGResult:
    answer: str
    input_tokens: int
    output_tokens: int

# Define the RAGBase class by inheriting from OriginalRAGBase and adapting it for our schema.
class RAGBase(OriginalRAGBase):

    # self,
    # index,
    # llm_client,
    # instructions=INSTRUCTIONS,
    # prompt_template=PROMPT_TEMPLATE,

    # Method to perform a search using the provided index.
    # Adapted for the filename/content schema.
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    # Method to build context string from search results.
    # Adapted for the filename/content schema.
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"Filename: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    # Method to interact with the LLM.
    # Overridden to use chat.completions.create and return the full response object for token counting.
    def llm(self, prompt):
        input_messages = [
            {"role": "system", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=input_messages
        )

        return response

    # Main RAG method to perform the entire RAG process.
    # Overridden to return a RAGResult with token counts.
    def rag(self, query):
        search_results = self.search(query)
        # print(search_results)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)


        # Extract the answer and token usage from the LLM response.
        answer = response.choices[0].message.content
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens

        return RAGResult(answer=answer, input_tokens=input_tokens, output_tokens=output_tokens)

In [ ]:
# Initialize RAGBase with the document index, Groq client, and specified model.
rag = RAGBase(
    index=index_sql,
    llm_client=client_groq,
    model="llama-3.3-70b-versatile"
)
# Run the RAG query.
result = rag.rag(query)
# Print the answer from the RAG result.
print(result.answer)
# Store the input tokens for Q3.
Q3_input_tokens = result.input_tokens
# Print the input tokens used.
print("Input tokens:", Q3_input_tokens)

The provided context includes multiple lessons and code snippets related to building an agentic RAG (Retrieval-Augmented Generator) pipeline. To answer your question about how the agentic loop keeps calling the model until it stops, we can look at the relevant code snippet:

```python
while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break
```

In this code, the agen

In [ ]:
print(rag.build_prompt)

<bound method RAGBase.build_prompt of <__main__.RAGBase object at 0x7bcce41875f0>>


In [ ]:
# Print the input and output tokens for Q3.
print(f"""Q3:
input tokens: {Q3_input_tokens}
output tokens: {result.output_tokens}""")

Q3:
input tokens: 7044
output tokens: 285


## Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not
what we do in practice. In practice we use libraries.

Let's use `VectorSearch` from minsearch and run a search for the following
query:

> What metric do we use to evaluate a search engine?

Which file is the `filename` of the first result?

* `02-vector-search/lessons/04-vector-search.md`
* `04-evaluation/lessons/05-search-metrics.md`
* `04-evaluation/lessons/13-llm-as-judge.md`
* `05-monitoring/lessons/04-metrics.md`

In [28]:
import numpy as np

# Assuming these are simple placeholder classes if the full .filters module is not provided
# and they are just for the structure of the custom VectorSearch.
class Filter:
    def __init__(self, field, value):
        self.field = field
        self.value = value

class FieldData:
    def __init__(self, key, value):
        self.key = key
        self.value = value

class VectorSearch:
    def __init__(self, keyword_fields: list = None):
        self.keyword_fields = keyword_fields if keyword_fields is not None else []
        self.vectors = None
        self.payload = None

    def fit(self, vectors: np.ndarray, payload: list):
        self.vectors = vectors
        self.payload = payload
        # For simplicity, we assume 'embedding' is present in chunks.
        # The 'embedding_field' is implicitly handled by providing the 'vectors' array directly.
        # The 'text_fields' are part of the 'payload' documents, not directly used for vector search here.

    def search(self, query_vector: np.ndarray, num_results: int = 5, filters: list = None):
        if self.vectors is None or self.payload is None:
            raise ValueError("VectorSearch index has not been fitted.")

        # Calculate dot product (cosine similarity since vectors are normalized)
        # query_vector might need to be reshaped if it's 1D, to (1, D) for dot product with (N, D)
        if query_vector.ndim == 1:
            query_vector = query_vector.reshape(1, -1)

        scores = self.vectors.dot(query_vector.T).flatten() # Result will be (N, 1), flatten to (N,)

        # Get top 'num_results' indices
        top_indices = np.argsort(scores)[::-1]

        results = []
        for idx in top_indices:
            doc = self.payload[idx]
            # Simple filter application (if needed, but Q4 doesn't specify filters for search)
            if filters:
                # This part is simplified and assumes basic key-value filtering for demonstration
                pass
            results.append(doc)
            if len(results) >= num_results:
                break
        return results

# Embed each chunk and add the embedding to the chunk dictionary
# This step was already performed in cell 'deb7569d' and 'chunks' and 'X' are already prepared.
# for chunk in chunks:
#     chunk['embedding'] = embedder.encode(chunk['content'])

# Initialize custom VectorSearch with keyword_fields
index_vector = VectorSearch(keyword_fields=["filename"])

# Fit the vector search index with the pre-computed embeddings (X) and chunks (payload)
index_vector.fit(vectors=X, payload=chunks)

# Define the query for Q4
q4_query = "What metric do we use to evaluate a search engine?"

# Embed the query using the existing embedder
q4_query_vector = embedder.encode(q4_query)

# Perform the vector search
vector_results_q4 = index_vector.search(
    query_vector=q4_query_vector,
    num_results=1
)

# Get the filename of the first result
q4_filename = vector_results_q4[0]['filename']

print(f"Q4: {q4_filename}")

Q4: 04-evaluation/lessons/05-search-metrics.md


## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index
the same chunks with `Index` from minsearch. Use `content` as a
text field.

Run both searches for this query:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the
vector results but not in the text results?

* `02-vector-search/lessons/01-intro.md`
* `02-vector-search/lessons/02-embeddings.md`
* `02-vector-search/lessons/08-pgvector.md`
* `03-orchestration/lessons/05-rag.md`


In [ ]:
import time
from sqlitesearch import TextSearchIndex

# Initialize a new TextSearchIndex for chunks
chunk_index_sql = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="lesson_llm_course_chunks.db" # Using a new database file for chunks
)

# Add each chunk to the new SQL index
for i, chunk in enumerate(chunks):
    chunk_index_sql.add(chunk)
    # print(f"Added chunk {i+1} from {chunk['filename']}...")
    # time.sleep(0.01) # Small delay to avoid overwhelming the system if many chunks

chunk_index_sql.close()
print("Done. Chunk index saved to lesson_llm_course_chunks.db")

Done. Chunk index saved to lesson_llm_course_chunks.db


In [ ]:
import os

file_path_chunks = 'lesson_llm_course_chunks.db'
file_size_bytes_chunks = os.path.getsize(file_path_chunks)
file_size_kb_chunks = file_size_bytes_chunks / 1024
file_size_mb_chunks = file_size_kb_chunks / 1024

print(f"File size of {file_path_chunks}: {file_size_mb_chunks:.2f} MB)")

File size of lesson_llm_course_chunks.db: 1.94 MB)


In [ ]:
# Initialize RAGBase with the chunked index, Groq client, and specified model.
rag_helper = RAGBase(
    index=chunk_index_sql,
    llm_client=client_groq,
    # model="llama-3.3-70b-versatile",
    model="openai/gpt-oss-120b"
)
# Run the RAG query with the chunked data.
result = rag_helper.rag(query)
# Print the answer from the RAG result.
print(result.answer)
# Store the input tokens for Q5.
Q5_input_tokens = result.input_tokens
# Print the input tokens used.
print("Input tokens:", Q5_input_tokens)
# Print the output tokens used.
print("Output tokens:", result.output_tokens)

The agentic loop is built around a simple **while‑True** loop that keeps sending the conversation history back to the model and only exits when the model stops asking for a tool.

**How it works**

1. **Start the loop** – `it = 1` and `while True:` begin an infinite loop.  
2. **Call the model** – `openai_client.responses.create(...)` is invoked with the current `messages` list (the full memory) and the list of available tools (e.g., `search_tool`).  
3. **Inspect the response** – The loop iterates over `response.output`.  
   * If an item’s `type` is **`function_call`**, the code:  
     * Prints the call, runs the corresponding function (`make_call(item)`),  
     * Appends the function’s output to `messages`, and  
     * Sets `has_function_calls = True`.  
   * If the item’s `type` is **`message`**, it prints the assistant’s reply (the final answer when no function calls follow).  
4. **Update the iteration counter** – `it = it + 1`.  
5. **Check the exit condition** – After proces

In [ ]:
# Calculate and print how many times fewer input tokens were used in Q5 compared to Q3.
print(f"Q5: {int(Q3_input_tokens/Q5_input_tokens)}x fewer")

Q5: 2x fewer


## Q6. Hybrid search

Both vector and text search have their strengths and weaknesses. Vector
search matches by meaning, so it finds relevant pages even when they use
words different from the query. But it can miss exact terms like names,
codes, or rare keywords. Text search is the opposite: it nails exact words
but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their
results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them
into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores
the raw scores from each method, which live on different scales and aren't
directly comparable. Instead, it looks only at the position of each
document in each list.

Every document scores by its position (`rank`, starting at 0) in each
list, and we sum the scores across lists with a constant `k = 60`:

```text
RRF(d) = sum over lists of  1 / (k + rank(d))
```

"Sum over lists" means we go through every ranked list and, for each list
where the document appears, add its `1 / (k + rank)` contribution. A
document found by both searches collects a score from each list, while one
found by only a single search collects just one.

The constant `k` controls how much the exact rank matters. A larger `k`
flattens the gap between positions, so the difference between rank 0 and
rank 5 counts for less. A smaller `k` does the opposite: it sharpens that
gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default.
You rarely need to tune it. Lower it when only the top results matter.
Raise it to reward documents that appear across many lists, even when they
never quite reach the top.

A document that ranks well in both lists ends up higher than one that's
only strong in a single list.

```python
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]
```

Now run the query `"How do I give the model access to tools?"`
with vector and text search and fuse the results with `rrf`:

```python
results = rrf([vector_results, text_results])
```

Which file is ranked first after RRF?

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/13-function-calling.md`
* `01-agentic-rag/lessons/14-agentic-loop.md`
* `01-agentic-rag/lessons/16-other-frameworks.md`


In [ ]:
import json

# Initialize a counter for search calls.
search_call_count = 0

# Define the search function, which simulates searching the course lessons.
# It increments a global counter and uses chunk_index to perform the actual search.
def search(query: str):
    """Search the course lessons for relevant content."""
    global search_call_count
    search_call_count += 1

    results = chunk_index_sql.search(query, num_results=5)
    return results

# Define available functions mapping their names to the actual function objects.
available_functions = {"search": search}

# Helper function to execute a function call (currently only 'search').
# It parses arguments, calls the function, and formats the output.
def make_call(tool_call, available_functions):
    """
    Выполняет функцию, запрошенную моделью, и возвращает сообщение
    в формате, который понимает Groq API.

    :param tool_call: Объект ChatCompletionMessageToolCall из ответа LLM
    :param available_functions: Словарь вида {'имя_функции': объект_функции_python}
    """
    function_name = tool_call.function.name
    # Аргументы приходят в виде строки JSON, их нужно распарсить
    function_args = json.loads(tool_call.function.arguments)

    print(f"Выполняю функцию: {function_name} с аргументами {function_args}")

    # Поиск функции в словаре доступных инструментов
    function_to_call = available_functions.get(function_name)

    if function_to_call:
        try:
            # Вызываем функцию
            function_response = function_to_call(**function_args)
        except Exception as e:
            function_response = f"Ошибка при выполнении функции: {str(e)}"
    else:
        function_response = f"Ошибка: Функция {function_name} не найдена."

    # Groq API требует, чтобы результат был строкой
    content = json.dumps(function_response, ensure_ascii=False)

    # Возвращаем словарь в формате сообщения 'tool'
    return {
        "role": "tool",
        "tool_call_id": tool_call.id, # ID, который прислала LLM для этого вызова
        "name": function_name,
        "content": content
    }

# Define the schema for the 'search' tool, describing its type, name, description, and parameters.
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the web for information",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            },
            "required": ["query"]
        }
    }
}


# Define the instructions for the developer role of the agent.
instructions = """You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."""

# Define the user's question for the agent.
question = "How does the agentic loop work, and how is it different from plain RAG?"

# Initialize the list of messages, starting with developer instructions and the user's question.
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question}
]

# Reset search call count and iteration counter.
search_call_count = 0
it = 1
last_answer = ""

# Initialize token usage counters.
total_input_tokens = 0
total_output_tokens = 0

# Start the agentic loop.
while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    # Send messages to the LLM with the search tool available.
    response = client_groq.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        tools=[search_tool],
        tool_choice="auto"
    )


    # Accumulate token usage from the LLM response.
    total_input_tokens += response.usage.prompt_tokens
    total_output_tokens += response.usage.completion_tokens

    # Extend the message history with the LLM's output.
    # 1. Ответ от LLM
    message = response.choices[0].message

    if message.tool_calls:
    # ОБЯЗАТЕЛЬНО: Сначала добавляем само сообщение ассистента с запросом на вызов
        messages.append(message)

        for tool_call in message.tool_calls:
            # 2. Выполняем вызов через наш хелпер
            tool_message = make_call(tool_call, available_functions)

            # 3. Добавляем результат выполнения в историю
            messages.append(tool_message)

    # Теперь можно делать следующий запрос к Groq, чтобы он проанализировал результат
    print(messages)
    break
    # Process each item in the LLM's output.
    for item in response.choices[0]:
        if item.type == "function":
            print(f"function_call: {item.name} {item.arguments}")
            # If a function call is detected, execute it using make_call.
            call_output = make_call(item, available_functions)
            # Add the output of the function call to the message history.
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            # If it's a message, print the assistant's response and store it.
            last_answer = item.content[0].text
            print(last_answer)

    # Increment iteration counter.
    it = it + 1
    # If no function calls were made in this iteration, break the loop.
    if has_function_calls == False:
        break

print(f"\n--- Number of Search Calls ---")
print(f"The agent called search {search_call_count} times")

# Example pricing for Groq's llama-3.3-70b-versatile (check Groq's official pricing for actual values)
# Assuming: Input token price = $0.70 per 1M tokens, Output token price = $0.80 per 1M tokens
price_per_million_input_tokens = 0.70
price_per_million_output_tokens = 0.80

# Calculate the estimated total cost based on token usage and example pricing.
total_cost = (
    (total_input_tokens / 1_000_000) * price_per_million_input_tokens +
    (total_output_tokens / 1_000_000) * price_per_million_output_tokens
)

print(f"\n--- Token Usage and Cost ---")
print(f"Total Input Tokens: {total_input_tokens}")
print(f"Total Output Tokens: {total_output_tokens}")
print(f"Estimated Total Cost: ${total_cost:.4f} (based on example pricing)")

iteration #1...
Выполняю функцию: search с аргументами {'query': 'agentic loop RAG agentic loop how does it work'}
[{'role': 'developer', 'content': "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."}, {'role': 'user', 'content': 'How does the agentic loop work, and how is it different from plain RAG?'}, ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user asks: "How does the agentic loop work, and how is it different from plain RAG?" We need to use the search tool. Let\'s search for "agentic loop" in context of LLM, RAG, agentic loop.', tool_calls=[ChatCompletionMessageToolCall(id='fc_fa96f115-d7a8-4d70-82cc-818f85351c72', function=Function(arguments='{"query":"agentic loop RAG agentic loop how does it work"}', name='search'), type='function')]), {'role': 'tool', 'tool_call_id': 'fc_fa96f115-d7a8-4d70-8

In [ ]:
# Install the toyaikit library, a small agent library for learning and experimentation.
%pip install toyaikit

In [ ]:
from toyaikit.tools import Tools

# Initialize ToyAIKit's Tools manager.
agent_tools = Tools()
# Register the 'search' function and its corresponding tool schema with agent_tools.
agent_tools.add_tool(search, search_tool)

print("agent_tools initialized and search function registered.")

agent_tools initialized and search function registered.


In [ ]:
# Display the tool schema that ToyAIKit generated or is using for the registered tools.
agent_tools.get_tools()

[{'type': 'function',
  'function': {'name': 'search',
   'description': 'Search the web for information',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string'}},
    'required': ['query']}}}]

In [ ]:
import json
import uuid # Added import for uuid
from toyaikit.llm import OpenAIClient, LLMClient, Message # Removed FunctionCall from import
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
from typing import List, Optional, Union
from groq import Groq

from pydantic import BaseModel, Field # Added Field for FunctionCall arguments

# Import OpenAI Pydantic models for compatibility (still useful for internal parsing from Groq)
from openai.types.chat.chat_completion import ChatCompletion, Choice
from openai.types.completion_usage import CompletionUsage as OpenAICompletionUsage # Renamed to avoid conflict
from openai.types.chat.chat_completion_message import ChatCompletionMessage
from openai.types.chat.chat_completion_message_tool_call import ChatCompletionMessageToolCall, Function as OpenAI_Function # Renamed to avoid conflict

# Define ToyAIKit-compatible Pydantic models if they are not directly importable
class ToyAIKitUsage(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int
    input_tokens: int = Field(alias="prompt_tokens") # Alias for toyaikit compatibility
    output_tokens: int = Field(alias="completion_tokens") # Alias for toyaikit compatibility

# Re-introducing custom FunctionCall model - This is now likely unnecessary if tool_calls is used
# class ToyAIKitFunctionCall(BaseModel):
#     name: str
#     arguments: str

# Define ToyAIKit-compatible ContentPart model
class ToyAIKitContentPart(BaseModel):
    text: str

# Assuming LLMClient and Tools are already defined in a previous cell and are in scope.

class GroqClient(LLMClient):
    def __init__(
        self,
        model: str = "openai/gpt-oss-120b",
        client=None,
        extra_kwargs: dict = None,
        api_key: Optional[str] = None,
    ):
        try:
            from groq import Groq
        except ImportError:
            raise ImportError(
                "Please run 'pip install groq' to use GroqClient"
            )

        self.model = model

        if client is None:
            self.client = Groq(api_key=api_key)
        else:
            self.client = client

        self.extra_kwargs = extra_kwargs or {}

    def send_request(
        self,
        chat_messages: List, # toyaikit passes this as a list of dictionaries
        tools: Tools = None,
        **kwargs,
    ):
        tools_list = []

        if tools is not None:
            tools_list = tools.get_tools()

        # Convert toyaikit's list of dicts to Groq-compatible message format
        groq_messages = []
        for msg in chat_messages:
            if isinstance(msg, dict):
                groq_messages.append(msg)
            elif hasattr(msg, 'model_dump'): # if it's a pydantic model
                # Handle toyaikit.llm.Message to Groq message format
                # If a toyaikit.llm.Message has tool_calls, we need to convert them
                msg_dict = msg.model_dump(exclude_unset=True)
                if 'tool_calls' in msg_dict and msg_dict['tool_calls']:
                    # Groq API expects 'tool_calls' directly in the message
                    groq_tool_calls = []
                    for tc in msg_dict['tool_calls']:
                        # Assuming tc is a dict or similar structure compatible with ChatCompletionMessageToolCall's constructor
                        # The 'function' part of the tool call needs to be an OpenAI_Function object
                        groq_tool_calls.append(ChatCompletionMessageToolCall(
                            id=tc.get('id', str(uuid.uuid4())),
                            function=OpenAI_Function(name=tc['function']['name'], arguments=tc['function']['arguments']),
                            type="function"
                        ))
                    msg_dict['tool_calls'] = groq_tool_calls
                    # Also ensure content is None for tool_call messages in Groq format
                    msg_dict['content'] = None # Groq expects None
                # Remove the custom 'type' if it's not expected by Groq for these messages
                # if 'type' in msg_dict and msg_dict['type'] != 'message':
                #     msg_dict.pop('type')
                groq_messages.append(msg_dict)
            else:
                groq_messages.append(dict(msg))

        args = dict(
            model=self.model,
            messages=groq_messages,
            tools=tools_list,
            **self.extra_kwargs,
        )

        if 'output_format' in kwargs:
            kwargs.pop('output_format')

        args.update(kwargs)

        groq_raw_response = self.client.chat.completions.create(**args)

        # --- Start of adaptation to toyaikit.llm.Response-compatible object ---

        # 1. Adapt usage object to ToyAIKitUsage
        toyaikit_usage = None
        if hasattr(groq_raw_response, 'usage') and groq_raw_response.usage:
            groq_usage = groq_raw_response.usage
            toyaikit_usage = ToyAIKitUsage(
                prompt_tokens=groq_usage.prompt_tokens,
                completion_tokens=groq_usage.completion_tokens,
                total_tokens=groq_usage.total_tokens
            )
            # Explicitly add input_tokens and output_tokens as attributes
            # This ensures they are directly accessible, satisfying toyaikit's runner.
            setattr(toyaikit_usage, 'input_tokens', groq_usage.prompt_tokens)
            setattr(toyaikit_usage, 'output_tokens', groq_usage.completion_tokens)

        # 2. Adapt choices/messages to a list of toyaikit.llm.Message objects
        toyaikit_messages_output = []
        if groq_raw_response.choices:
            groq_choice = groq_raw_response.choices[0]
            groq_message = groq_choice.message

            # Handle tool calls first (if any)
            if groq_message.tool_calls:
                toyaikit_tool_calls_list = []
                for tc in groq_message.tool_calls: # tc is groq.types.chat.chat_completion_message_tool_call.ChatCompletionMessageToolCall
                    toyaikit_tool_calls_list.append(
                        ChatCompletionMessageToolCall( # Use the imported OpenAI type for compatibility
                            id=tc.id,
                            function=OpenAI_Function(
                                name=tc.function.name,
                                arguments=tc.function.arguments
                            ),
                            type="function" # This is the type of the tool call itself
                        )
                    )

                # Append the message with tool_calls
                toyaikit_messages_output.append(
                    Message(
                        id=str(uuid.uuid4()), # Generate a unique ID for the message
                        role="assistant", # Tool calls are initiated by the assistant
                        # For tool calls, content should be an empty list as per toyaikit.llm.Message validation.
                        # However, DisplayingRunnerCallback.on_message expects content[0].
                        # Provide a minimal content part to satisfy this, returning an empty string.
                        content=[ToyAIKitContentPart(text="")],
                        tool_calls=toyaikit_tool_calls_list, # Pass the list of tool calls
                        type="message", # Must be 'message' even for function calls as per ValidationError
                        model=self.model,
                        usage=toyaikit_usage.model_dump() if toyaikit_usage else None # Convert to dict
                    )
                )

            # Handle content message (if any content is present)
            if groq_message.content:
                toyaikit_messages_output.append(
                    Message(
                        id=str(uuid.uuid4()), # Generate a unique ID
                        role=groq_message.role,
                        content=[ToyAIKitContentPart(text=groq_message.content)], # Wrap content in ContentPart
                        type="message", # Explicitly set type to "message" for text content
                        model=self.model,
                        usage=toyaikit_usage.model_dump() if toyaikit_usage else None # Convert to dict
                    )
                )

        # 3. Construct a toyaikit-compatible response object
        # This class will mimic toyaikit.llm.OpenAIClient.Response
        class ToyAIKitCompatibleResponse(BaseModel):
            output: List[Message]
            usage: Optional[ToyAIKitUsage] = None # Use our new ToyAIKitUsage here

        return ToyAIKitCompatibleResponse(output=toyaikit_messages_output, usage=toyaikit_usage)

    @property
    def responses(self):
        """Compatibility layer for RAGBase and other tools expecting .responses.create"""
        return self

    def create(self, model: str = None, input: List = None, **kwargs):
        """Compatibility method for the 'responses' API."""
        return self.send_request(
            chat_messages=input,
            model=model or self.model,
            **kwargs
        )

# Initialize an IPythonChatInterface for interactive chat display in the notebook.
chat_interface = IPythonChatInterface()
# Initialize a DisplayingRunnerCallback to visualize model messages and tool calls.
callback = DisplayingRunnerCallback(chat_interface)

# Initialize an OpenAIClient for ToyAIKit, using the same model as in the agentic loop
# and explicitly passing the Groq API key for authentication.
llm_client_toyaikit = GroqClient(model="openai/gpt-oss-120b", api_key=api_key_groq) # Pass the api_key_groq here
# llm_client_toyaikit =
# Initialize the OpenAIResponsesRunner, which orchestrates the agentic loop.
# It takes registered tools, developer instructions, the chat interface, and the LLM client.
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client_toyaikit
)

print("Chat interface and runner initialized.")

Chat interface and runner initialized.


In [ ]:
# Execute the agentic loop using the runner with the defined question and callback.
# The 'question' variable is taken from Q6's definition.
result = runner.loop(
    prompt=question, # Using the 'question' from Q6
    callback=callback,
)

ValidationError: 12 validation errors for Message
content.0.TextBlock
  Input should be a valid dictionary or instance of TextBlock [type=model_type, input_value=ToyAIKitContentPart(text=''), input_type=ToyAIKitContentPart]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type
content.0.ThinkingBlock
  Input should be a valid dictionary or instance of ThinkingBlock [type=model_type, input_value=ToyAIKitContentPart(text=''), input_type=ToyAIKitContentPart]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type
content.0.RedactedThinkingBlock
  Input should be a valid dictionary or instance of RedactedThinkingBlock [type=model_type, input_value=ToyAIKitContentPart(text=''), input_type=ToyAIKitContentPart]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type
content.0.ToolUseBlock
  Input should be a valid dictionary or instance of ToolUseBlock [type=model_type, input_value=ToyAIKitContentPart(text=''), input_ty

In [ ]:
# Print the total number of times the 'search' function was explicitly called by the agent.
print(f"The agent called search {search_call_count} times.")

The agent called search 1 times.


In [ ]:
# Display the complete history of messages, including system, user, assistant, function calls, and function call outputs, from the agent's run.
print(result.all_messages)

AttributeError: 'RAGResult' object has no attribute 'all_messages'

In [ ]:
print("\n--- Analyzing Tool Calls History ---")
# Iterate through all messages in the result to analyze the structure of tool calls.
for i, message in enumerate(result.all_messages):
    # Check if the message type is a function call.
    if message.type == "function_call":
        print(f"Message {i}: Function Call - Name: {message.name}, Arguments: {message.arguments}")
    # Check if the message type is a function call output.
    elif message.type == "function_call_output":
        print(f"Message {i}: Function Call Output - Call ID: {message.call_id}, Output: {message.output}")
    # Optionally, you can also print assistant messages.
    elif message.type == "message":
        # print(f"Message {i}: Assistant Message - Content: {message.content[0].text}")
        pass


--- Analyzing Tool Calls History ---


AttributeError: 'RAGResult' object has no attribute 'all_messages'

# ToyAIKit

Видео: [Смотреть этот урок](https://www.youtube.com/watch?v=PQpQOR3Un3w&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

Написанный вручную агентный цикл из предыдущего урока полезен для обучения, но он довольно однообразен. Каждый раз при создании нового агента вам пришлось бы писать один и тот же цикл `while`, одну и ту же обработку вызовов функций и управление сообщениями.

`ToyAIKit` инкапсулирует этот паттерн, позволяя вам сосредоточиться на инструментах, промптах и поведении. Мы построили его вместе на одном из воркшопов DataTalks.Club некоторое время назад. Он делает то же самое, что и наш ручной цикл, но с меньшим количеством шаблонного кода. Если вы заглянете в код `runners`, вы найдете там тот же цикл `while True`, который мы писали сами.

Я использую его здесь намеренно, потому что не хочу выбирать победителя среди промышленных фреймворков. `ToyAIKit` маленький и легко читается, поэтому, если что-то сломается, вы сможете увидеть, что именно произошло. Это делает его удобным для разработки и отладки локально перед переходом в продакшн.

Одно предостережение: `ToyAIKit` — это библиотека для обучения и экспериментов, она НЕ предназначена для использования в реальных проектах (production). Мы используем ее, потому что она минималистична и наглядна.

## Настройка

Установите библиотеку:

```bash
uv add toyaikit
```

Импортируйте необходимые классы:

```python
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
```

## Регистрация инструмента

Мы регистрируем нашу функцию `search` вместе со схемой из предыдущих уроков:

```python
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
```

## Автоматическая генерация схемы в ToyAIKit

Писать схему вручную утомительно, и мы не хотим делать это для каждой функции. На самом деле, это и не обязательно.

Если мы добавим аннотации типов и строку документации (docstring) к функции `search`, `ToyAIKit` прочитает их и сам составит схему за нас:

```python
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )
```

Затем зарегистрируйте ее без передачи схемы:

```python
agent_tools = Tools()
agent_tools.add_tool(search)
```

Вы можете посмотреть, что сгенерировал `ToyAIKit`:

```python
agent_tools.get_tools()
```

На выходе вы получите ту же JSON-схему, которую мы писали вручную в уроке про вызов функций. `ToyAIKit` создал ее на основе докстринга и аннотации типа.

Любой современный агентный фреймворк использует этот прием. Он считывает типизированную функцию Python с документацией и строит на ее основе схему. Так работают OpenAI Agents SDK, PydanticAI, LangChain и Google ADK. Вы пишете инструмент, а фреймворк сам понимает, как его описать.

## Интерфейс чата и раннер (runner)

Создайте интерфейс чата и колбэк, а затем постройте раннер:

```python
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)
```

`chat_interface` отвечает за отображение в ноутбуке. `callback` отрисовывает сообщения модели и вызовы инструментов по мере их возникновения. Раннер запускает агентный цикл — тот самый `while True`, который мы писали вручную. Он отправляет сообщения, выполняет вызовы функций, добавляет результаты инструментов обратно и повторяет процесс до тех пор, пока модель не закончит.

Мы намеренно выбрали здесь `gpt-5.4-mini`. Без нее `ToyAIKit` использует более простую и быструю модель по умолчанию, которая не так надежно следует инструкциям.

## Запуск одного промпта

Запустите один запрос:

```python
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)
```

Мы специально использовали опечатку «Olama». Агент выполняет поиск, получает плохие результаты, а затем пробует еще раз с «Ollama». Механизм восстановления такой же, как в ручном цикле. Вывод в ноутбуке выглядит гораздо приятнее: каждый вызов инструмента и сообщение отображаются по очереди, так что вы можете изучить каждый результат поиска.

`result` — это объект `LoopResult`, содержащий `all_messages` (всю историю диалога), количество токенов и `cost` (стоимость, рассчитанную на основе использования токенов).

## Стоимость и токены

Посмотрите, сколько стоил этот вызов:

```python
result.cost
```

Это полезно при разработке — особенно для многоходовых агентов, где один запрос может вызвать несколько обращений к модели. В ручном цикле вам приходилось считать это самим. Фреймворк делает это за вас автоматически.

Вы также можете просмотреть полную историю сообщений:

```python
result.all_messages
```

Это просто список — тот же список `messages`, который мы вели вручную.

## Продолжение диалога

Возьмите сообщения из предыдущего результата и передайте их как `previous_messages` при следующем вызове `loop`:

```python
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)
```

Раннер продолжит с того места, где остановился предыдущий вызов, используя тот же агентный цикл и расширенную историю. Модель поймет, что под «другой моделью» подразумевается Ollama, так как она видит предыдущий ход в памяти. Без этой истории она бы понятия не имела, о чем идет речь.

## Интерактивный чат

Для работы в режиме чата запустите встроенный цикл ввода:

```python
runner.run()
```

Вводите вопросы и получайте ответы. Введите «stop», чтобы выйти.

[← Агентный цикл](14-agentic-loop.md) | [Другие фреймворки →](16-other-frameworks.md)


In [ ]:
from typing import List, Optional

from openai import OpenAI
from pydantic import BaseModel

from openai.types.chat.chat_completion import ChatCompletion
from openai.types.chat.parsed_chat_completion import ParsedChatCompletion
from openai.types.responses.response import Response
from openai.types.responses.parsed_response import ParsedResponse

from anthropic.types import Message, RawMessageStopEvent

from toyaikit.tools import Tools


class LLMClient:
    def send_request(self, chat_messages: List, tools: Tools = None):
        raise NotImplementedError("Subclasses must implement this method")

class GroqClient(LLMClient):
    def __init__(
        self,
        model: str = "openai/gpt-oss-120b",
        client=None,
        extra_kwargs: dict = None,
        api_key: Optional[str] = None, # Added api_key parameter
    ):
        try:
            from groq import Groq
        except ImportError:
            raise ImportError(
                "Please run 'pip install groq' to use GroqClient"
            )

        self.model = model

        if client is None:
            # Pass the api_key directly to the Groq client
            self.client = Groq(api_key=api_key)
        else:
            self.client = client

        self.extra_kwargs = extra_kwargs or {}

    def send_request(
        self,
        chat_messages: List,
        tools: Tools = None,
        **kwargs,
    ):
        tools_list = []

        if tools is not None:
            tools_list = tools.get_tools()

        args = dict(
            model=self.model,
            messages=chat_messages,
            tools=tools_list,
            **self.extra_kwargs,
        )

        # Remove 'output_format' if present, as Groq API does not support it directly
        # or it is handled differently by the Groq client. ToyAIKit passes it by default.
        if 'output_format' in kwargs:
            kwargs.pop('output_format')

        args.update(kwargs)

        return self.client.chat.completions.create(**args)

    @property
    def responses(self):
        """Compatibility layer for RAGBase and other tools expecting .responses.create"""
        return self

    def create(self, model: str = None, input: List = None, **kwargs):
        """Compatibility method for the 'responses' API."""
        return self.send_request(
            chat_messages=input,
            model=model or self.model,
            **kwargs
        )

class GroqChatCompletionsClient(LLMClient):
    def __init__(
        self,
        model: str = "openai/gpt-oss-120b",
        client=None,
        extra_kwargs: dict = None,
    ):
        try:
            from groq import Groq
        except ImportError:
            raise ImportError(
                "Please run 'pip install groq' to use GroqChatCompletionsClient"
            )

        self.model = model

        if client is None:
            self.client = Groq()
        else:
            self.client = client

        self.extra_kwargs = extra_kwargs or {}

    def convert_single_tool(self, tool):
        """
        Convert a single OpenAI tool/function API dict to Groq-compatible format.
        """
        return {
            "type": "function",
            "function": {
                "name": tool["name"],
                "description": tool["description"],
                "parameters": tool["parameters"],
            },
        }

    def convert_api_tools_to_chat_functions(self, api_tools):
        """
        Convert a list of OpenAI API tools to Groq-compatible format.
        """
        chat_functions = []

        for tool in api_tools:
            converted = self.convert_single_tool(tool)
            chat_functions.append(converted)

        return chat_functions

    def send_request(
        self,
        chat_messages: List,
        tools: Tools = None,
        **kwargs,
    ):
        tools_list = []

        if tools is not None:
            tools_requests_format = tools.get_tools()
            tools_list = self.convert_api_tools_to_chat_functions(
                tools_requests_format,
            )

        args = dict(
            model=self.model,
            messages=chat_messages,
            tools=tools_list,
            **self.extra_kwargs,
        )
        args.update(kwargs)

        return self.client.chat.completions.create(**args)

In [ ]:
import json
import uuid
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Callable, Generic, TypeVar

from openai.types.chat.chat_completion_function_message_param import (
    ChatCompletionFunctionMessageParam,
)
from openai.types.chat.chat_completion_system_message_param import (
    ChatCompletionSystemMessageParam,
)
from openai.types.chat.chat_completion_user_message_param import (
    ChatCompletionUserMessageParam,
)
from openai.types.responses.easy_input_message import EasyInputMessage
from openai.types.responses.response_function_tool_call import ResponseFunctionToolCall
from pydantic import BaseModel

from toyaikit.chat.interface import ChatInterface
from toyaikit.llm import LLMClient
from toyaikit.pricing import CostInfo, PricingConfig, TokenUsage
from toyaikit.tools import Tools

# T must be either a str or a (subclass)
# instance of pydantic BaseModel
T = TypeVar("T", str, BaseModel)


def _get_tool_call_output(call_result) -> str:
    """Extract output from tool call result, handling both dict and object types."""
    if isinstance(call_result, dict):
        return call_result.get("output", "")
    return getattr(call_result, "output", "")


@dataclass
class LoopResult(Generic[T]):
    new_messages: list
    all_messages: list
    tokens: TokenUsage
    cost: CostInfo | None
    last_message: T


class RunnerCallback(ABC):
    """Abstract base class for different chat runners."""

    @abstractmethod
    def on_function_call(self, function_call: dict, result: str):
        """
        Called when a function call is made.
        """
        pass

    @abstractmethod
    def on_message(self, message: dict):
        """
        Called when a message is received.
        """
        pass

    @abstractmethod
    def on_reasoning(self, reasoning: str):
        """
        Called when a reasoning is received.
        """
        pass

    @abstractmethod
    def on_response(self, response):
        pass


class ChatRunner(ABC):
    """Abstract base class for different chat runners."""

    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_type: BaseModel = None
    ) -> LoopResult:
        """
        Do one tool-call loop.
        """
        pass

    @abstractmethod
    def run(self, previous_messages: list = None) -> LoopResult:
        """
        Repeast tool call loops until user asks to stop
        """
        pass


class DisplayingRunnerCallback(RunnerCallback):
    def __init__(self, chat_interface: ChatInterface):
        self.chat_interface = chat_interface

    def on_function_call(self, function_call, result):
        self.chat_interface.display_function_call(
            function_call.name,
            function_call.arguments,
            result,
        )

    def on_message(self, message):
        self.chat_interface.display_response(message)

    def on_reasoning(self, reasoning):
        self.chat_interface.display_reasoning(reasoning)

    def on_response(self, response):
        self.chat_interface.display("-> Response received")


class BaseToolUsingRunner(ChatRunner):
    """Base class for runners that use tools and LLM clients.

    Provides common initialization and run() method implementation.
    Subclasses only need to implement the loop() method.
    """

    def __init__(
        self,
        tools: Tools = None,
        developer_prompt: str = "You're a helpful assistant.",
        chat_interface: ChatInterface = None,
        llm_client: LLMClient = None,
        pricing_config: PricingConfig = None,
    ):
        self.tools = tools
        self.developer_prompt = developer_prompt
        self.chat_interface = chat_interface
        self.llm_client = llm_client
        self.displaying_callback = DisplayingRunnerCallback(chat_interface)
        self.pricing_config = pricing_config or PricingConfig()

    @abstractmethod
    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        """Execute one tool-call loop. Must be implemented by subclasses."""
        pass

    def run(
        self,
        previous_messages: list = None,
        stop_criteria: Callable = None,
    ) -> LoopResult:
        """Repeat tool-call loops until user asks to stop."""
        chat_messages = self._initialize_messages(previous_messages)

        total_input_tokens = 0
        total_output_tokens = 0
        last_message_text = ""

        while True:
            user_input = self.chat_interface.input()
            if user_input.lower() == "stop":
                self.chat_interface.display("Chat ended.")
                break

            loop_result = self.loop(
                prompt=user_input,
                previous_messages=chat_messages,
                callback=self.displaying_callback,
            )

            chat_messages.extend(loop_result.new_messages)
            total_input_tokens += loop_result.tokens.input_tokens
            total_output_tokens += loop_result.tokens.output_tokens
            last_message_text = loop_result.last_message

            if stop_criteria and stop_criteria(loop_result.new_messages):
                break

        combined_cost = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )
        combined_tokens = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        return LoopResult(
            new_messages=chat_messages,
            all_messages=chat_messages,
            tokens=combined_tokens,
            cost=combined_cost,
            last_message=last_message_text,
        )

    @abstractmethod
    def _initialize_messages(self, previous_messages: list = None) -> list:
        """Initialize chat messages. Must be implemented by subclasses."""
        pass


class OpenAIResponsesRunner(BaseToolUsingRunner):
    """Runner for OpenAI responses API."""

    def _initialize_messages(self, previous_messages: list = None) -> list:
        if previous_messages is None or len(previous_messages) == 0:
            return [
                EasyInputMessage(
                    role="developer",
                    content=self.developer_prompt,
                )
            ]
        return list(previous_messages)  # Return a copy

    def loop(
        self,
        prompt: str,
        previous_messages: list[dict] = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        chat_messages = []
        prev_messages_len = 0

        if previous_messages is None or len(previous_messages) == 0:
            chat_messages.append(
                EasyInputMessage(
                    role="developer",
                    content=self.developer_prompt,
                )
            )
        else:
            chat_messages.extend(previous_messages)
            prev_messages_len = len(previous_messages)

        chat_messages.append(
            EasyInputMessage(
                role="user",
                content=prompt,
            )
        )

        total_input_tokens = 0
        total_output_tokens = 0

        while True:
            response = self.llm_client.send_request(
                chat_messages=chat_messages,
                tools=self.tools,
                output_format=output_format,
            )
            if callback:
                callback.on_response(response)

            if hasattr(response, "usage") and response.usage:
                total_input_tokens += response.usage.input_tokens
                total_output_tokens += response.usage.output_tokens

            has_function_calls = False

            chat_messages.extend(response.output)

            for entry in response.output:
                if entry.type == "function_call":
                    result = self.tools.function_call(entry)
                    chat_messages.append(result)
                    if callback:
                        callback.on_function_call(entry, result['output'])
                    has_function_calls = True

                elif entry.type == "message":
                    if callback:
                        callback.on_message(entry.content[0].text)

            if not has_function_calls:
                break

        cost_info = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )

        token_usage = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        new_messages = chat_messages[prev_messages_len:]

        last_message_text = ""
        last_message = None
        for entry in reversed(response.output):
            if entry.type == "message":
                last_message_text = entry.content[0].text
                if output_format:
                    last_message = output_format.model_validate_json(last_message_text)
                else:
                    last_message = last_message_text
                break

        return LoopResult(
            new_messages=new_messages,
            all_messages=chat_messages,
            tokens=token_usage,
            cost=cost_info,
            last_message=last_message,
        )


class OpenAIAgentsSDKRunner(ChatRunner):
    """Runner for OpenAI Agents SDK."""

    def __init__(self, chat_interface: ChatInterface, agent):
        try:
            from agents import Runner
        except ImportError:
            raise ImportError(
                "Please run 'pip install openai-agents' to use this feature"
            )

        self.agent = agent
        self.runner = Runner()
        self.chat_interface = chat_interface

    async def run(self) -> None:
        from agents import SQLiteSession

        session_id = f"chat_session_{uuid.uuid4().hex[:8]}"
        session = SQLiteSession(session_id)

        while True:
            user_input = self.chat_interface.input()
            if user_input.lower() == "stop":
                self.chat_interface.display("Chat ended.")
                break

            result = await self.runner.run(
                self.agent, input=user_input, session=session
            )

            func_calls = {}
            for ni in result.new_items:
                raw = ni.raw_item
                if ni.type == "tool_call_item":
                    func_calls[raw.call_id] = raw

            for ni in result.new_items:
                raw = ni.raw_item

                if ni.type == "handoff_call_item":
                    raw = ni.raw_item
                    self.chat_interface.display(f"handoff: {raw.name}")

                if ni.type == "handoff_output_item":
                    self.chat_interface.display(
                        f"handoff: {ni.target_agent.name} -> {ni.source_agent.name} successful"
                    )

                if ni.type == "tool_call_output_item":
                    call_id = raw["call_id"]
                    if call_id not in func_calls:
                        self.chat_interface.display(
                            f"error: cannot find the call parameters for {call_id=}"
                        )
                    else:
                        func_call = func_calls[call_id]
                        self.chat_interface.display_function_call(
                            func_call.name, func_call.arguments, raw["output"]
                        )

                if ni.type == "message_output_item":
                    md = raw.content[0].text
                    self.chat_interface.display_response(md)


class PydanticAIRunner(ChatRunner):
    """Runner for Pydantic AI."""

    def __init__(self, chat_interface: ChatInterface, agent):
        self.chat_interface = chat_interface
        self.agent = agent

    async def run(self) -> None:
        message_history = []

        while True:
            user_input = self.chat_interface.input()
            if user_input.lower() == "stop":
                self.chat_interface.display("Chat ended.")
                break

            result = await self.agent.run(
                user_prompt=user_input,
                message_history=message_history
            )

            messages = result.new_messages()
            tool_calls = {}

            for m in messages:
                for part in m.parts:
                    kind = part.part_kind

                    if kind == "text":
                        self.chat_interface.display_response(part.content)

                    elif kind == "tool-call":
                        tool_calls[part.tool_call_id] = part

                    elif kind == "tool-return":
                        call = tool_calls.get(part.tool_call_id)

                        raw_result = part.content
                        if raw_result is None:
                            result_str = ""
                        elif isinstance(raw_result, str):
                            result_str = raw_result
                        else:
                            result_str = json.dumps(raw_result, indent=2, default=str)

                        self.chat_interface.display_function_call(
                            call.tool_name,
                            json.dumps(call.args),
                            result_str
                        )

            message_history.extend(messages)



class OpenAIChatCompletionsRunner(BaseToolUsingRunner):
    """Runner for OpenAI chat completions API."""

    def _initialize_messages(self, previous_messages: list = None) -> list:
        if previous_messages is None or len(previous_messages) == 0:
            return [
                ChatCompletionSystemMessageParam(
                    role="system",
                    content=self.developer_prompt
                )
            ]
        return list(previous_messages)  # Return a copy

    @staticmethod
    def convert_function_output_to_tool_message(data):
        return ChatCompletionFunctionMessageParam(
            role="tool",
            tool_call_id=data["call_id"],
            content=data["output"],
        )

    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        chat_messages = []
        prev_messages_len = 0

        if previous_messages is None or len(previous_messages) == 0:
            chat_messages.append(
                ChatCompletionSystemMessageParam(
                    role="system",
                    content=self.developer_prompt
                )
            )
        else:
            chat_messages.extend(previous_messages)
            prev_messages_len = len(previous_messages)

        user_message = ChatCompletionUserMessageParam(
            role="user",
            content=prompt
        )
        chat_messages.append(user_message)

        total_input_tokens = 0
        total_output_tokens = 0

        while True:
            reponse = self.llm_client.send_request(
                chat_messages, self.tools, output_format
            )

            if callback:
                callback.on_response(reponse)

            if reponse.usage:
                total_input_tokens += reponse.usage.prompt_tokens
                total_output_tokens += reponse.usage.completion_tokens

            first_choice = reponse.choices[0]
            message_response = first_choice.message
            chat_messages.append(message_response)

            if hasattr(message_response, "reasoning_content"):
                reasoning = (message_response.reasoning_content or "").strip()
                if reasoning != "" and callback:
                    callback.on_reasoning(reasoning)

            content = (message_response.content or "").strip()
            if content != "" and callback:
                callback.on_message(content)

            calls = []

            if hasattr(message_response, "tool_calls"):
                calls = message_response.tool_calls

            if calls is None:
                break

            if len(calls) == 0:
                break

            for call in calls:
                function_call = ResponseFunctionToolCall(
                    type="function_call",
                    name=call.function.name,
                    arguments=call.function.arguments,
                    call_id=call.id,
                )

                call_result = self.tools.function_call(function_call)
                call_result = self.convert_function_output_to_tool_message(call_result)

                chat_messages.append(call_result)

                if callback:
                    content_val = getattr(call_result, "content", None)
                    if content_val is None and isinstance(call_result, dict):
                        content_val = call_result.get("content")
                    callback.on_function_call(function_call, content_val)

        cost = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )

        token_usage = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        new_messages = chat_messages[prev_messages_len:]

        last_message_text = (message_response.content or "").strip()
        if output_format:
            last_message = output_format.model_validate_json(last_message_text)
        else:
            last_message = last_message_text

        return LoopResult(
            new_messages=new_messages,
            all_messages=chat_messages,
            tokens=token_usage,
            cost=cost,
            last_message=last_message,
        )


class AnthropicMessagesRunner(BaseToolUsingRunner):
    """Runner for Anthropic Messages API."""

    def _initialize_messages(self, previous_messages: list = None) -> list:
        if previous_messages is None or len(previous_messages) == 0:
            return [{
                "role": "system",
                "content": self.developer_prompt
            }]
        return list(previous_messages)  # Return a copy

    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        chat_messages = []
        prev_messages_len = 0

        if previous_messages is None or len(previous_messages) == 0:
            chat_messages.append({
                "role": "system",
                "content": self.developer_prompt
            })
        else:
            chat_messages.extend(previous_messages)
            prev_messages_len = len(previous_messages)

        chat_messages.append({
            "role": "user",
            "content": prompt
        })

        total_input_tokens = 0
        total_output_tokens = 0

        while True:
            response = self.llm_client.send_request(
                chat_messages=chat_messages,
                tools=self.tools,
                output_format=output_format,
            )

            if callback:
                callback.on_response(response)

            # Track token usage
            if hasattr(response, "usage") and response.usage:
                total_input_tokens += response.usage.input_tokens
                total_output_tokens += response.usage.output_tokens

            # Process the response
            assistant_message = {
                "role": "assistant",
                "content": response.content
            }
            chat_messages.append(assistant_message)

            has_tool_calls = False
            text_content = []

            for block in response.content:
                if block.type == "text":
                    text_content.append(block.text)
                    if callback:
                        callback.on_message(block.text)

                elif block.type == "tool_use":
                    has_tool_calls = True
                    function_call = ResponseFunctionToolCall(
                        type="function_call",
                        name=block.name,
                        arguments=json.dumps(block.input),
                        call_id=block.id,
                    )

                    call_result = self.tools.function_call(function_call)
                    result_output = _get_tool_call_output(call_result)

                    # Anthropic expects tool results in a user message with tool_result blocks
                    tool_result_message = {
                        "role": "tool",
                        "tool_call_id": block.id,
                        "content": result_output
                    }
                    chat_messages.append(tool_result_message)

                    if callback:
                        callback.on_function_call(function_call, result_output)

            if not has_tool_calls:
                break

        cost_info = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )

        token_usage = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        new_messages = chat_messages[prev_messages_len:]

        # Extract last message text
        last_message_text = ""
        last_message = None
        if text_content:
            last_message_text = "".join(text_content)
            if output_format:
                last_message = output_format.model_validate_json(last_message_text)
            else:
                last_message = last_message_text

        return LoopResult(
            new_messages=new_messages,
            all_messages=chat_messages,
            tokens=token_usage,
            cost=cost_info,
            last_message=last_message,
        )

In [ ]:
🚀 Module 1 of LLM Zoomcamp by @DataTalksClub complete!

Just finished Module 1 - Agentic RAG. Learned how to:

✅ Build a RAG system from scratch in plain Python
✅ Index and search documents with minsearch
✅ Chunk long documents for better retrieval
✅ Turn the RAG pipeline into an agent with function calling

Here's my homework solution: <LINK>

Following along with this amazing free course - who else is learning to build with LLMs?

You can sign up here: https://github.com/DataTalksClub/llm-zoomcamp/